In [ ]:
MIN_EDGE_WEIGHT = 15
TOP_N_NODES = 200
RANDOM_SEED = 42

In [ ]:
import os
import pandas as pd
import networkx as nx
import igraph as ig
import leidenalg
import matplotlib.pyplot as plt
from itertools import combinations
from dotenv import load_dotenv
from transformers import pipeline

In [ ]:
load_dotenv()

VIDEO_ID = os.getenv("VIDEO_ID1")
data_path = os.getenv("DATA_PATH")
output_path = os.path.dirname(data_path.rstrip("/"))

print(f"VIDEO_ID: {VIDEO_ID} | MIN_EDGE_WEIGHT: {MIN_EDGE_WEIGHT} | TOP_N_NODES: {TOP_N_NODES}")
print(f"Saídas serão salvas em: {output_path}")

In [ ]:
# carregamento dos comentários lematizados
lemmatized_path = os.getenv("LEMMATIZED_PATH")

df = pd.read_csv(lemmatized_path)
comments = df["comment"].fillna("").astype(str).tolist()
print(f"Comentários lematizados carregados: {len(comments)}")

In [ ]:
# construção do grafo de co-ocorrência
G = nx.Graph()

print("Populando grafo...")
for i, comment in enumerate(comments):
    if i % 5000 == 0: print(f"  Processando linha {i}...")

    palavras = list(set(comment.split()))  # tokens já lematizados, deduplica por comentário

    for w1, w2 in combinations(palavras, 2):
        if G.has_edge(w1, w2):
            G[w1][w2]["weight"] += 1
        else:
            G.add_edge(w1, w2, weight=1)

print(f"Grafo bruto: {G.number_of_nodes()} nós, {G.number_of_edges()} arestas")

In [ ]:
# filtragem: threshold de aresta, top N nós, giant component
print(f"Cortando arestas com peso < {MIN_EDGE_WEIGHT}...")
edges_to_remove = [(u, v) for u, v, w in G.edges(data="weight") if w < MIN_EDGE_WEIGHT]
G.remove_edges_from(edges_to_remove)
G.remove_nodes_from(list(nx.isolates(G)))

if G.number_of_nodes() > TOP_N_NODES:
    print(f"Filtrando para os top {TOP_N_NODES} nós mais centrais...")
    top_nodes = [n for n, d in sorted(G.degree, key=lambda x: x[1], reverse=True)[:TOP_N_NODES]]
    G = G.subgraph(top_nodes).copy()
    G.remove_nodes_from(list(nx.isolates(G)))

if len(G) > 0:
    largest_cc = max(nx.connected_components(G), key=len)
    G = G.subgraph(largest_cc).copy()

# normalização dos pesos
if G.number_of_edges() > 0:
    max_weight = max(G[u][v]["weight"] for u, v in G.edges())
    for u, v in G.edges():
        G[u][v]["norm_weight"] = G[u][v]["weight"] / max_weight

print(f"Após filtragem: {G.number_of_nodes()} nós, {G.number_of_edges()} arestas")

In [ ]:
# detecção de comunidades com Leiden
deg_cent = nx.degree_centrality(G)

nodes = list(G.nodes())
node_index = {n: i for i, n in enumerate(nodes)}

ig_graph = ig.Graph(n=len(nodes))
ig_edges = [(node_index[u], node_index[v]) for u, v in G.edges()]
ig_weights = [G[u][v]["norm_weight"] for u, v in G.edges()]
ig_graph.add_edges(ig_edges)

partition = leidenalg.find_partition(
    ig_graph,
    leidenalg.ModularityVertexPartition,
    weights=ig_weights,
    seed=RANDOM_SEED
)

partition_dict = {}
for community_id, members in enumerate(partition):
    for member_idx in members:
        partition_dict[nodes[member_idx]] = community_id

mod = partition.modularity
print(f"Modularidade (Leiden): {mod:.4f}")
print(f"Número de comunidades: {len(partition)}")

for node in G.nodes():
    G.nodes[node]['community_id'] = partition_dict[node]

In [ ]:
# rotulação automática das comunidades com LLM
print("Carregando modelos de rotulação...")
gerador = pipeline("text2text-generation", model="google/flan-t5-base")
classificador = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

df_nodes = pd.DataFrame({
    "node": nodes,
    "degree_centrality": [deg_cent[n] for n in nodes],
    "community_id": [partition_dict[n] for n in nodes]
})

comunidades_ids = sorted(df_nodes["community_id"].unique())
rotulos_gerados = {}

for com_id in comunidades_ids:
    top_nos = (
        df_nodes[df_nodes["community_id"] == com_id]
        .sort_values("degree_centrality", ascending=False)
        .head(20)["node"]
        .tolist()
    )
    termos_str = ", ".join(top_nos)

    # Passo A: Text Generation — rótulo livre
    prompt = f"These are the most central terms from a community in a network of Brazilian political debate comments: {termos_str}. What is the central theme? Answer in up to 5 words."
    resultado = gerador(prompt, max_new_tokens=20)[0]["generated_text"]
    rotulo_livre = resultado.strip()

    # Passo B: Zero-Shot Classification — validação
    hipoteses = [rotulo_livre, "ataques políticos", "pauta econômica", "mobilização ideológica", "discurso moralizante"]
    resultado_zs = classificador(termos_str, candidate_labels=hipoteses)
    rotulo_final = resultado_zs["labels"][0]

    rotulos_gerados[com_id] = rotulo_final
    print(f"Comunidade {com_id}: '{rotulo_livre}' → validado como '{rotulo_final}'")

print("\nRótulos finais:")
for com_id, rotulo in rotulos_gerados.items():
    n = len(df_nodes[df_nodes["community_id"] == com_id])
    print(f"  {com_id}: {rotulo} ({n} nós)")

for node in G.nodes():
    G.nodes[node]["community_label"] = rotulos_gerados.get(partition_dict[node], str(partition_dict[node]))

In [ ]:
# análise exploratória
print("=== Métricas da Rede de Co-ocorrência ===")
print(f"Nós (palavras): {G.number_of_nodes()}")
print(f"Arestas: {G.number_of_edges()}")
print(f"Densidade: {nx.density(G):.6f}")
print(f"Número de comunidades: {len(rotulos_gerados)}")
print(f"Modularidade (Leiden): {mod:.4f}")

print("\n=== Comunidades ===")
for com_id, rotulo in rotulos_gerados.items():
    n = len(df_nodes[df_nodes["community_id"] == com_id])
    print(f"  [{com_id}] {rotulo}: {n} nós ({100*n/G.number_of_nodes():.1f}%)")

# distribuição de grau
degrees = [d for _, d in G.degree()]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(degrees, bins=50, color='steelblue', edgecolor='white')
ax.set_xlabel("Grau")
ax.set_ylabel("Frequência")
ax.set_title(f"Distribuição de Grau — Rede de Co-ocorrência ({VIDEO_ID})")
plt.tight_layout()
plt.savefig(os.path.join(output_path, f"distribuicao_grau_coocorrencia_{VIDEO_ID}.png"), dpi=150)
plt.show()

In [ ]:
# exportação
caminho_gexf = os.path.join(output_path, f"grafo_coocorrencia_{VIDEO_ID}.gexf")
nx.write_gexf(G, caminho_gexf)
print(f"Grafo exportado: {caminho_gexf}")

df_nodes["community_label"] = df_nodes["community_id"].map(rotulos_gerados)
caminho_csv = os.path.join(output_path, f"comunidades_coocorrencia_{VIDEO_ID}.csv")
df_nodes[["node", "degree_centrality", "community_id", "community_label"]].to_csv(
    caminho_csv, index=False, encoding='utf-8-sig'
)
print(f"Tabela de comunidades exportada: {caminho_csv}")